In [13]:
import cv2 as cv
import numpy as np
import pandas as pd
from tqdm import tqdm

In [14]:
openFace = np.load('./data/openFace.npy').astype(np.int32)
openFaceArea = np.load('./data/openFaceArea.npy')
closedFace = np.load('./data/closedFace.npy').astype(np.int32)
closedFaceArea = np.load('./data/closedFaceArea.npy')

In [15]:
# 分别定义左右眼的面积数组，并借助 tqdm 遍历面部特征数组
def processData(face,faceArea):
    leftEyeData = np.empty([0])
    rightEyeData = np.empty([0])

    for i in tqdm(range(face.shape[0])):
        oneFace = face[i]
        # 计算左眼面积及占比
        leftEye = oneFace[36:42]
        leftArea = cv.contourArea(leftEye)
        leftEyeData = np.append(leftEyeData,leftArea / faceArea[i])
        # 计算右眼面积及占比
        rightEye = oneFace[42:48]
        rightArea = cv.contourArea(rightEye)
        rightEyeData = np.append(rightEyeData,rightArea / faceArea[i])
                                 
    # 维度扩增
    leftEyeData = np.expand_dims(leftEyeData,axis=1)
    rightEyeData = np.expand_dims(rightEyeData,axis=1)

    # 按列合并
    return np.concatenate((leftEyeData,rightEyeData),axis=1)

In [16]:
# 处理睁眼数据集
openFaceData = processData(openFace,openFaceArea)

100%|██████████| 1226/1226 [00:00<00:00, 39327.71it/s]


In [17]:
openFaceData = np.insert(openFaceData,2,values=0,axis=1)

In [18]:
print(openFaceData[0:5])

[[0.00496942 0.00441726 0.        ]
 [0.004842   0.00458716 0.        ]
 [0.0029304  0.00311355 0.        ]
 [0.00488447 0.00420489 0.        ]
 [0.00873016 0.00842491 0.        ]]


In [19]:
# 处理闭眼数据集
closedFaceData = processData(closedFace,closedFaceArea)
closedFaceData = np.insert(closedFaceData,2,values=1,axis=1)

100%|██████████| 1002/1002 [00:00<00:00, 46295.36it/s]


In [20]:
# 合并睁眼和闭眼数据集
faceData = np.concatenate((openFaceData,closedFaceData),axis=0)

In [21]:
# 打乱数据集和标签
permutation = np.random.permutation(faceData.shape[0])
faceData = faceData[permutation,:]

In [22]:
# 拆分数据集
vfold_size = int(faceData.shape[0]/100*(0.2*100))
trainData = faceData[vfold_size:,:]
evalData = faceData[0:vfold_size,:]

In [23]:
def exportData(data,path):
    faceDataFrame = pd.DataFrame(data)
    faceDataFrame[[2]] = faceDataFrame[[2]].astype(int)
    faceDataFrame.columns = ['leftEye','rightEye','isTired']
    faceDataFrame.to_csv(path,index=False)

In [24]:
exportData(trainData,'./data/train.csv')
exportData(evalData,'./data/eval.csv')